# Importing 

Modules imported for the Neural Network:
- __Numpy__: For complex calculations
- __Rasterio__: For TIFF image processing 
- __PyTorch__: Model functionality, and Dataset creation

In [8]:
import os
import numpy as np
import rasterio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# Setting up the Dataset
We create a class that inherits from Dataset. When created, this class:
- Loads samples from samples folder on initialization
- When calling __\_\_getitem\_\___:
    - The function takes an index as an argument
    - It loads the sample and label images with the file name in the index
    - Normalizes the sample image using Rasterio
    - Classifies the label image into one of three classes: No vegetation, low amount of vegetation, and high amount of vegetation
    - Converts images into PyTorch tensors.

In [9]:
class VegetationDataset(Dataset):
    def __init__(self, samples_dir, labels_dir):
        self.samples_dir = samples_dir
        self.labels_dir = labels_dir
        self.files = os.listdir(samples_dir)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_name = self.files[idx]

        label_name = file_name.replace("_img_", "_ndvi_") 
        label_path = os.path.join(self.labels_dir, label_name)

        # Load sample image
        with rasterio.open(os.path.join(self.samples_dir, file_name)) as src:
            img = src.read().astype(np.float32)  # (3, H, W)

        # Normalization of sample image
        img = img / 255.0

        # Load label image
        with rasterio.open(os.path.join(self.labels_dir, label_name)) as src:
            label = src.read(1).astype(np.float32)  # (H, W)

        # Convert label → classes
        # Adjust thresholds if needed
        label_class = np.zeros_like(label, dtype=np.int64)
        label_class[label < 120] = 0
        label_class[(label >= 120) & (label < 170)] = 1
        label_class[label >= 170] = 2

        return torch.tensor(img), torch.tensor(label_class)

# Loading images

Dataset initialization.

In [10]:
samples_dir = "../../samples"
labels_dir = "../../labels"

dataset = VegetationDataset(samples_dir, labels_dir)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

# Convolutional Neural Network Class

This class creates a Convolutional Neural Network that inherits from the nn.Module module. On initialization:
- Initializes nn.Module object
- Sets up model layers using Sequential method: 
    - First the input layer, with 3 input channels (RGB layers) and 16 output channels (new features)
    - Then the processing layers, with 16 input channels and 32 output channels
    - Finally the output layer, with 32 input channels and 3 output channels (three possible classes)
    - We apply ReLU to each layer to add non-linearity.
- We also define the flow of data through the network with the forward() method.

In [11]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, 3, kernel_size=1)  # 3 classes
        )

    def forward(self, x):
        return self.net(x)

# Model initialization

- model: Initializes SimpleCNN object
- criterion: Loss function. Measures how wrong the preditions are.
- optimizer: Updates weights to reduce loss.

In [12]:
model = SimpleCNN()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Model Training

The following code includes all steps to complete model training on the entire dataset. What it does:
- Defines the number of epochs (number of times the model will train over the entire dataset)
- Tracks loss through each epoch by using the loss function
- For each image/label pair in the dataset
    - Pass each image through the model
    - Calculates loss by comparing outputs with labels
    - Clears old gradients
    - loss.backward() determines how each weight contributed to the error
    - optimizer.step() adjusts model pparameters using gradients
    - total_loss is updated.


In [13]:
epochs = 10

for epoch in range(epochs):
    total_loss = 0

    for imgs, labels in loader:
        outputs = model(imgs)  # (B, 3, H, W)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}: Loss = {total_loss:.4f}")

Epoch 0: Loss = 112.2476
Epoch 1: Loss = 90.7675
Epoch 2: Loss = 88.4841
Epoch 3: Loss = 86.6446
Epoch 4: Loss = 86.0476
Epoch 5: Loss = 85.3623
Epoch 6: Loss = 82.5715
Epoch 7: Loss = 80.9305
Epoch 8: Loss = 78.2575
Epoch 9: Loss = 75.4985


# Model Evaluation

We define the evaluate function. This function:
- defines the number of correctly predictede pixels and the total number of pixels evaluated.
- turns off gradient computation (evaluation without learning, faster evaluation that uses less memory)
- iterates through dataset to:
    - processes image through defined model
    - compares the output with the most probable class
    - update correct and total variables

In [17]:
def evaluate(model, loader, num_classes=3):
    # Put the model in evaluation mode to freeze layers like Dropout/BatchNorm
    model.eval() 
    
    # Initialize tensors to track metrics for each of the 3 classes
    tp = torch.zeros(num_classes)
    fp = torch.zeros(num_classes)
    fn = torch.zeros(num_classes)
    
    # Keep tracking overall pixel accuracy
    correct = 0 
    total = 0   

    with torch.no_grad():
        for imgs, labels in loader:
            
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)
            
            # Overall accuracy tracking
            correct += (preds == labels).sum().item()
            total += labels.numel()
            
            # Calculate TP, FP, FN for each class
            for c in range(num_classes):
                true_c = (labels == c)
                pred_c = (preds == c)
                
                # True Positives
                tp[c] += torch.sum(true_c & pred_c).item()
                
                # False Positives
                fp[c] += torch.sum(~true_c & pred_c).item()
                
                # False Negatives: 
                fn[c] += torch.sum(true_c & ~pred_c).item()

    # Calculate metrics (adding a tiny epsilon to prevent division by zero)
    epsilon = 1e-7
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)
    f1 = 2 * (precision * recall) / (precision + recall + epsilon)
    
    # Print Overall Accuracy
    print(f"Overall Pixel Accuracy: {(correct / total) * 100:.2f}% \n")
    
    # Print Per-Class Metrics
    for c in range(num_classes):
        print(f"--- Class {c} ---")
        print(f"Precision: {precision[c].item():.4f}")
        print(f"Recall:    {recall[c].item():.4f}")
        print(f"F1 Score:  {f1[c].item():.4f}\n")
        
    # Print Macro-Average Metrics (Average across all classes)
    print("--- Macro Averages ---")
    print(f"Mean Precision: {precision.mean().item():.4f}")
    print(f"Mean Recall:    {recall.mean().item():.4f}")
    print(f"Mean F1 Score:  {f1.mean().item():.4f}")


evaluate(model, loader)

c:\Adversarial\Learning_Path\nn_venv\Lib\site-packages\rasterio\__init__.py:379: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


Overall Pixel Accuracy: 75.12% 

--- Class 0 ---
Precision: 0.0000
Recall:    0.0000
F1 Score:  0.0000

--- Class 1 ---
Precision: 0.9539
Recall:    0.3861
F1 Score:  0.5498

--- Class 2 ---
Precision: 0.7139
Recall:    0.9931
F1 Score:  0.8307

--- Macro Averages ---
Mean Precision: 0.5559
Mean Recall:    0.4597
Mean F1 Score:  0.4601
